<a href="https://colab.research.google.com/github/justdoit0430/newcrawl/blob/main/2news.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import requests
import json
from datetime import datetime, timedelta
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font
from urllib.parse import quote, unquote
import re
import time
import logging
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import files
import threading

# 로깅 설정
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# 네이버 API 설정
client_id = "W82Xhl54o5knFsACcgV6"
client_secret = "4oRhzO3DqM"

headers = {
    "X-Naver-Client-Id": client_id,
    "X-Naver-Client-Secret": client_secret
}

# 날짜 설정 - 최근 1주일
current_date = datetime.now()
one_week_ago = current_date - timedelta(days=7)

final_news_list = []
# 스레드 안전을 위한 락 추가
news_list_lock = threading.Lock()

def clean_html(raw_html):
    cleanr = re.compile('<.*?>')
    cleantext = re.sub(cleanr, '', raw_html)
    cleantext = unquote(cleantext)
    return cleantext

def format_date(date_str):
    try:
        return datetime.strptime(date_str, '%a, %d %b %Y %H:%M:%S %z').strftime('%Y-%m-%d')
    except:
        return datetime.now().strftime('%Y-%m-%d')

def fetch_news_for_keyword(keyword, retries=3):
    encoded_search_word = quote(keyword)
    local_news_list = []
    local_seen_links = set()  # 각 키워드별로 로컬 set 사용

    for start in range(1, 1001, 100):
        api_url = (
            f"https://openapi.naver.com/v1/search/news.json"
            f"?query={encoded_search_word}&start={start}&display=100&sort=date"
        )

        try:
            response = requests.get(api_url, headers=headers, timeout=10)
            if response.status_code == 429:
                time.sleep(2)
                continue
            response.raise_for_status()
        except Exception as e:
            if retries > 0:
                time.sleep(1)
                return fetch_news_for_keyword(keyword, retries - 1)
            logging.error(f"'{keyword}' 에러: {e}")
            return local_news_list

        news_data = response.json()
        items = news_data.get('items', [])

        if not items:
            break

        added_count = 0
        duplicate_count = 0
        old_news_count = 0

        for item in items:
            news_date_str = format_date(item['pubDate'])
            news_date = datetime.strptime(news_date_str, '%Y-%m-%d').date()

            # 1주일 이내 뉴스만 수집
            if news_date >= one_week_ago.date():
                news_title = clean_html(item['title'])
                news_link = item['link']
                news_description = clean_html(item.get('description', ''))

                # 로컬 set으로 중복 체크 (키워드 내에서만)
                if news_link not in local_seen_links:
                    local_seen_links.add(news_link)
                    local_news_list.append([
                        keyword,
                        news_date_str,
                        news_title,
                        news_link,
                        news_description
                    ])
                    added_count += 1
                else:
                    duplicate_count += 1
            else:
                old_news_count += 1

        logging.info(f"'{keyword}' - {start}~{start+99}: {added_count}건 추가, {duplicate_count}건 중복, {old_news_count}건 1주일 초과")

        # 1주일 초과 뉴스가 많아지면 해당 키워드 수집 종료
        if old_news_count > 80:
            logging.info(f"'{keyword}' - 1주일 초과 뉴스가 많아 수집 종료")
            break

        # API 호출 간격 조절
        time.sleep(0.05)

    return local_news_list

def fetch_news_parallel(keywords):
    with ThreadPoolExecutor(max_workers=3) as executor:
        future_to_keyword = {executor.submit(fetch_news_for_keyword, keyword): keyword for keyword in keywords}
        for future in as_completed(future_to_keyword):
            keyword = future_to_keyword[future]
            try:
                news_items = future.result()
                # 스레드 안전하게 리스트에 추가
                with news_list_lock:
                    final_news_list.extend(news_items)
                logging.info(f"✅ '{keyword}' 수집 완료: {len(news_items)}건")
            except Exception as e:
                logging.error(f"❌ '{keyword}' 실패: {e}")

# 키워드 목록
search_keywords = [
    "유미코아", "테슬라", "BYD", "GM", "기아", "ev3", "전고체", "르노",
    "CATL", "삼성SDI", "LG에너지솔루션", "SK온", "파나소닉", "LG화학",
    "토요타", "스텔란티스", "BMW", "FEOC", "포스코퓨처엠", "양극재",
    "음극재", "엘앤에프", "에코프로", "전구체", "전기차", "ESS",
    "데이터센터", "전력"
]

# 수집 정보 출력
print(f"🔍 총 {len(search_keywords)}개 키워드로 최근 1주일간 뉴스 수집")
print(f"📅 수집 기간: {one_week_ago.strftime('%Y-%m-%d')} ~ {current_date.strftime('%Y-%m-%d')}")
print(f"🔑 검색 키워드: {', '.join(search_keywords)}")
print(f"🚀 {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} 뉴스 수집을 시작합니다.\n")

# 수집 실행
fetch_news_parallel(search_keywords)

# 결과 확인
print(f"\n📊 총 수집된 뉴스: {len(final_news_list)}건")

# 키워드별 통계
if final_news_list:
    df_temp = pd.DataFrame(final_news_list, columns=["Keyword", "Date", "Title", "Link", "Description"])
    keyword_stats = df_temp['Keyword'].value_counts()
    print(f"\n📈 키워드별 뉴스 수:")
    for keyword, count in keyword_stats.items():
        print(f"  {keyword}: {count}건")

    # 날짜별 통계
    date_stats = df_temp['Date'].value_counts().sort_index(ascending=False)
    print(f"\n📅 날짜별 뉴스 수:")
    for date, count in date_stats.items():
        print(f"  {date}: {count}건")

# 데이터 가공 및 저장
if final_news_list:
    df = pd.DataFrame(final_news_list, columns=["Keyword", "Date", "Title", "Link", "Description"])

    # 최종 중복 제거 (링크 기준으로만)
    before_count = len(df)
    df = df.drop_duplicates(subset=['Link'], keep='first')
    after_count = len(df)

    print(f"\n🔄 최종 중복 제거:")
    print(f"  제거 전: {before_count}건")
    print(f"  제거 후: {after_count}건")
    print(f"  중복 제거: {before_count - after_count}건")

    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values(by='Date', ascending=False)
    df['Date'] = df['Date'].dt.strftime('%Y-%m-%d')

    # 파일 저장
    file_name = f"battery_news_1week_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx"

    # 엑셀 저장 시 컬럼 너비 조정
    with pd.ExcelWriter(file_name, engine='openpyxl') as writer:
        df.to_excel(writer, index=False, sheet_name='뉴스목록')
        worksheet = writer.sheets['뉴스목록']

        # 컬럼 너비 조정
        worksheet.column_dimensions['A'].width = 15  # Keyword
        worksheet.column_dimensions['B'].width = 12  # Date
        worksheet.column_dimensions['C'].width = 80  # Title
        worksheet.column_dimensions['D'].width = 15  # Link
        worksheet.column_dimensions['E'].width = 100 # Description

    # 하이퍼링크 적용
    try:
        wb = load_workbook(file_name)
        ws = wb.active

        # 헤더 스타일 적용
        for col in range(1, 6):
            cell = ws.cell(row=1, column=col)
            cell.font = Font(bold=True)

        # 제목에 하이퍼링크 적용
        for row in range(2, ws.max_row + 1):
            title_cell = ws.cell(row=row, column=3)
            link = ws.cell(row=row, column=4).value
            if link:
                title_cell.hyperlink = link
                title_cell.font = Font(color='0000FF', underline='single')

        wb.save(file_name)
        print(f"\n✅ 파일 생성 완료: {file_name}")

        # Colab에서 파일 다운로드
        files.download(file_name)

    except Exception as e:
        print(f"❌ 엑셀 편집 중 오류: {e}")
        files.download(file_name)
else:
    print("❌ 수집된 데이터가 없습니다.")

print(f"\n🏁 {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} 뉴스 수집이 완료되었습니다.")


🔍 총 28개 키워드로 최근 1주일간 뉴스 수집
📅 수집 기간: 2026-04-13 ~ 2026-04-20
🔑 검색 키워드: 유미코아, 테슬라, BYD, GM, 기아, ev3, 전고체, 르노, CATL, 삼성SDI, LG에너지솔루션, SK온, 파나소닉, LG화학, 토요타, 스텔란티스, BMW, FEOC, 포스코퓨처엠, 양극재, 음극재, 엘앤에프, 에코프로, 전구체, 전기차, ESS, 데이터센터, 전력
🚀 2026-04-20 22:54:36 뉴스 수집을 시작합니다.


📊 총 수집된 뉴스: 13975건

📈 키워드별 뉴스 수:
  데이터센터: 998건
  테슬라: 997건
  SK온: 996건
  기아: 996건
  전력: 993건
  LG에너지솔루션: 991건
  전기차: 980건
  ESS: 943건
  에코프로: 932건
  BMW: 791건
  삼성SDI: 676건
  LG화학: 558건
  르노: 524건
  BYD: 484건
  GM: 346건
  양극재: 296건
  토요타: 296건
  전고체: 247건
  포스코퓨처엠: 192건
  CATL: 155건
  음극재: 120건
  엘앤에프: 109건
  파나소닉: 107건
  전구체: 103건
  스텔란티스: 75건
  ev3: 60건
  FEOC: 7건
  유미코아: 3건

📅 날짜별 뉴스 수:
  2026-04-21: 686건
  2026-04-20: 4903건
  2026-04-19: 999건
  2026-04-18: 490건
  2026-04-17: 1869건
  2026-04-16: 1573건
  2026-04-15: 1163건
  2026-04-14: 1314건
  2026-04-13: 978건

🔄 최종 중복 제거:
  제거 전: 13975건
  제거 후: 10152건
  중복 제거: 3823건

✅ 파일 생성 완료: battery_news_1week_20260420_225521.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🏁 2026-04-20 22:55:27 뉴스 수집이 완료되었습니다.
